In [0]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import AzureError

from src.config.secrets import get_secret

def get_storage_config():

    storage_account = get_secret(
        "AZURE-STORAGE-ACCOUNT",
        "AZURE_STORAGE_ACCOUNT"
    )

    storage_key = get_secret(
        "AZURE-STORAGE-KEY",
        "AZURE_STORAGE_KEY"
    )

    container_name = get_secret(
        "AZURE-CONTAINER-BRONZE",
        "AZURE_CONTAINER_BRONZE"
    )

    missing = []

    if not storage_account:
        missing.append("AZURE-STORAGE-ACCOUNT")

    if not storage_key:
        missing.append("AZURE-STORAGE-KEY")

    if not container_name:
        missing.append("AZURE-CONTAINER-BRONZE")

    if missing:
        raise ValueError(
            f"Segredos ausentes: {', '.join(missing)}"
        )

    return {
        "storage_account": storage_account,
        "storage_key": storage_key,
        "container_name": container_name
    }


def get_blob_service_client():
    config = get_storage_config()
    account_url = (
        f"https://{config['storage_account']}.blob.core.windows.net")

    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=config["storage_key"]
    )

    container_client = (
        blob_service_client.get_container_client(
            config["container_name"]
        )
    )

    container_client.get_container_properties()

    # print("✅ Azure Storage conectado")

    return (
        blob_service_client,
        config["container_name"]
    )

